In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import math

自注意力机制

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, Q, K, V, mask=None):
        d_k = Q.size(-1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        # 处理mask
        if mask is not None:
            scores = scores.masked_fill(mask==0, float('-inf'))

        attn = self.softmax(scores)
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)

        return out, attn

多头注意力机制

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.self_attention = SelfAttention(dropout)

        # 融合多头
        self.fc = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

        self.norm = nn.LayerNorm(d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)

        Q = self.W_q(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        out, attn = self.self_attention(Q, K, V, mask)

        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_k)

        out = self.fc(out)

        out = self.dropout(out)

        return self.norm(out + q), attn

前馈神经网络

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        out = self.fc2(self.dropout(torch.relu(self.fc1(x))))
        return self.norm(out + x)

编码层（EncoderLayer）

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, src, src_mask=None):
        out, _ = self.attention(src, src, src, src_mask)
        out = self.ffn(out)

        return out

解码层（DecoderLayer）

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, tgt, memory, tgt_mask, memory_mask):
        out, _ = self.self_attention(tgt, tgt, tgt, tgt_mask)
        out, _ = self.cross_attention(out, memory, memory, memory_mask)
        out = self.ffn(out)

        return out

位置编码

In [ ]:
class PositionEmbedding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        # 列
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # 缩放因子
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model))

        # 位置信息
        # 偶数
        pe[:, 0::2] = torch.sin(position * div_term)
        # 奇数
        pe[:, 1::2] = torch.cos(position * div_term)

        # 增加batch维
        pe = pe.unsqueeze(0)

        # 注册为buffer
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        seq_len = x.size(1)

        return x + self.pe[:, :seq_len, :]

编码器（Encoder）

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, num_layers, max_len=5000, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = PositionEmbedding(d_model, max_len)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)
        ])

    def forward(self, src, src_mask=None):
        out = self.embedding(src) * math.sqrt(self.embedding.embedding_dim)
        out = self.pos_embedding(out)

        for layer in self.layers:
            out = layer(out, src_mask)
        
        return out

解码器（Decoder）

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, num_layers, max_len=5000, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = PositionEmbedding(d_model, max_len)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        out = self.embedding(tgt) * math.sqrt(self.embedding.embedding_dim)
        out = self.pos_embedding(out)
        for layer in self.layers:
            out = layer(out, memory, tgt_mask, memory_mask)

        out = self.fc_out(out)

        return out

Mask-Attention

In [ ]:
def MaskAttention(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return mask==0

Transformer架构

In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, tgt_vocab_size, d_model=512, n_heads=8, d_ff=2048, num_encoder_layers=6, num_decoder_layers=6, max_len=5000, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(vocab_size, d_model, n_heads, d_ff, num_encoder_layers, max_len, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, n_heads, d_ff, num_decoder_layers, max_len, dropout)
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None):
        memory = self.encoder(src, src_mask)
        out = self.decoder(tgt, memory, tgt_mask, memory_mask)

        return out